In [ ]:
import cv2

frame = cv2.imread('./data/balloons.jpg')

# Grayscale
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# Resize
resized = cv2.resize(gray, (224, 224))
normalized = resized / 255.0


print(normalized.shape)
print(normalized.min(), normalized.max())


In [ ]:
import torch

tensor = torch.tensor(normalized, dtype=torch.float32)
tensor = tensor.permute(2, 0, 1)   # (H, W, 3) → (3, H, W)
tensor = tensor.unsqueeze(0)        # (3, H, W) → (1, 3, H, W)

print(tensor.shape)  # torch.Size([1, 3, 224, 224])

In [ ]:
import torch.nn as nn

class DQN(nn.Module):
    def __init__(self, state_size=4, num_actions=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_actions),
        )

    def forward(self, x):
        return self.net(x)

model = DQN()

In [ ]:
import gymnasium as gym
env = gym.make("CartPole-v1")
state, _ = env.reset()
print(state)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# Hyperparameters
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01
gamma = 0.99          # discount factor
batch_size = 32
lr = 1e-4

model = DQN(num_actions=2)
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.MSELoss()
memory = deque(maxlen=10000)  # replay buffer

def select_action(state):
    if random.random() < epsilon:
        return random.randint(0, 1)
    with torch.no_grad():
        return model(state).argmax().item()

def train():
    if len(memory) < batch_size:
        return

    batch = random.sample(memory, batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)

    states      = torch.tensor(states, dtype=torch.float32)
    next_states = torch.tensor(next_states, dtype=torch.float32)
    actions     = torch.tensor(actions)
    rewards     = torch.tensor(rewards, dtype=torch.float32)
    dones       = torch.tensor(dones, dtype=torch.float32)

    current_q   = model(states).gather(1, actions.unsqueeze(1)).squeeze()
    next_q      = model(next_states).max(1).values
    target_q    = rewards + gamma * next_q * (1 - dones)

    loss = loss_fn(current_q, target_q.detach())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Training loop
for episode in range(1000):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        action = select_action(state_tensor)
        next_state, reward, done, truncated, _ = env.step(action)
        done = done or truncated

        memory.append((state, action, reward, next_state, done))
        train()
        state = next_state
        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 100 == 0:
        print(f"Episode {episode}, reward: {total_reward}, epsilon: {epsilon:.2f}")